In [ ]:
# ============================================================
# 1. Konfigurasi & Helper
# ============================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, date
import time, random, re, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

# ── Konfigurasi ───────────────────────────────────────────────
TANGGAL_MULAI   = date(2024, 1, 1)
TANGGAL_AKHIR   = date.today()
SAVE_DIR        = r"C:\Users\Asus\skripsi"
TARGET_NORMAL   = 300   # per sumber × 5 sumber = 1500
TARGET_DISTORSI = 400   # per sumber × 4 sumber = 1600
MAX_WORKERS     = 10
MAX_HALAMAN     = 20
DELAY           = (0.2, 0.6)

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"📅 Rentang : {TANGGAL_MULAI} → {TANGGAL_AKHIR}")
print(f"📁 Simpan  : {SAVE_DIR}")

UA_POOL = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36",
]

lock = Lock()
def log(m):
    with lock: print(m)

def headers(ref=None):
    h = {"User-Agent": random.choice(UA_POOL),
         "Accept": "text/html,*/*;q=0.8", "Accept-Language": "id-ID,id;q=0.9",
         "Connection": "keep-alive", "Upgrade-Insecure-Requests": "1"}
    if ref: h["Referer"] = ref
    return h

def get_soup(url, ref=None, timeout=15, retries=2):
    s = requests.Session()
    for i in range(retries + 1):
        try:
            r = s.get(url, headers=headers(ref), timeout=timeout)
            if r.status_code == 200:
                return BeautifulSoup(r.text, "lxml")
            if r.status_code in (403, 429):
                time.sleep((2 ** i) * random.uniform(1, 2))
            elif r.status_code in (404, 410):
                return None
        except:
            time.sleep(random.uniform(0.5, 1))
    return None

def parse_tanggal(teks):
    if not teks: return None
    teks = re.sub(r'\s+', ' ', teks.strip())
    bulan = {
        "januari":1,"februari":2,"maret":3,"april":4,"mei":5,"juni":6,
        "juli":7,"agustus":8,"september":9,"oktober":10,"november":11,"desember":12,
        "jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
        "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12
    }
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})', teks)
    if m:
        try: return date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        except: pass
    m = re.search(r'(\d{1,2})\s+(\w+)\s+(\d{4})', teks, re.I)
    if m:
        b = bulan.get(m.group(2).lower())
        if b:
            try: return date(int(m.group(3)), b, int(m.group(1)))
            except: pass
    return None

def dalam_rentang(teks):
    d = parse_tanggal(teks)
    return d is not None and TANGGAL_MULAI <= d <= TANGGAL_AKHIR

def get_tgl(soup):
    for sel in ["meta[property='article:published_time']", "time",
                "meta[name='publishdate']", "meta[name='date']"]:
        el = soup.select_one(sel)
        if el:
            return el.get("content") or el.get("datetime") or el.get_text(strip=True)
    return ""

def kumpul_links(urls, pola, home, limit):
    from urllib.parse import urlparse
    links, seen = [], set()
    for base in urls:
        if len(links) >= limit: break
        kosong = 0
        for page in range(1, MAX_HALAMAN + 1):
            if len(links) >= limit: break
            url = base if page == 1 else (
                f"{base}&page={page}" if "?" in base else f"{base}?page={page}")
            soup = get_soup(url, ref=home)
            if not soup:
                kosong += 1
                if kosong >= 2: break
                continue
            baru = 0
            for a in soup.find_all('a', href=True):
                h = a['href']
                if h.startswith('/'):
                    p = urlparse(base)
                    h = f"{p.scheme}://{p.netloc}{h}"
                if pola(h) and h not in seen:
                    links.append(h); seen.add(h); baru += 1
            kosong = 0 if baru else kosong + 1
            if kosong >= 2: break
            time.sleep(random.uniform(*DELAY))
    return links

print("✅ Helper siap")


# ============================================================
# 2. Scraping Normal
# ============================================================

def tambah_artikel(articles, judul, isi, tgl, sumber, url, max_n):
    if len(articles) >= max_n: return False
    if not judul or not isi or len(isi) < 100: return False
    if not dalam_rentang(tgl): return False
    articles.append({"judul": judul.strip(), "isi": isi.strip(),
                     "tanggal": tgl, "sumber": sumber, "url": url, "label": ""})
    print(f"  ✓ [{len(articles)}/{max_n}] {judul.strip()[:65]}")
    return True

def scrape_normal(nama, no, total, categories, pola_href, parse_judul, parse_isi, parse_tgl_fn, max_n=TARGET_NORMAL):
    print(f'\n[{no}/{total}] Scraping {nama} (target {max_n})...')
    links = kumpul_links(categories, pola_href, categories[0], limit=max_n * 5)
    print(f'  [INFO] {len(links)} link ditemukan')
    articles = []
    for url in links:
        if len(articles) >= max_n: break
        soup = get_soup(url)
        if not soup: continue
        try:
            judul = soup.select_one(parse_judul)
            isi   = " ".join(p.get_text(strip=True) for p in soup.select(parse_isi))
            tgl   = soup.select_one(parse_tgl_fn)
            tambah_artikel(articles,
                judul.get_text(strip=True) if judul else "",
                isi,
                tgl.get_text(strip=True) if tgl else "",
                nama, url, max_n)
        except: pass
        time.sleep(random.uniform(0.3, 0.8))
    return articles

semua_normal = []

semua_normal += scrape_normal(
    "Detik.com", 1, 5,
    ["https://news.detik.com/berita","https://news.detik.com/berita-jawa-tengah",
     "https://news.detik.com/berita-jawa-timur","https://news.detik.com/berita-jawa-barat",
     "https://news.detik.com/internasional","https://finance.detik.com/berita-ekonomi-bisnis",
     "https://hot.detik.com","https://sport.detik.com","https://health.detik.com",
     "https://www.detik.com/terpopuler"],
    lambda h: 'detik.com' in h and '/d/' in h,
    "h1.detail__title", "div.detail__body-text p", "div.detail__date"
)

semua_normal += scrape_normal(
    "Kompas.com", 2, 5,
    ["https://nasional.kompas.com","https://regional.kompas.com","https://money.kompas.com",
     "https://tekno.kompas.com","https://health.kompas.com","https://bola.kompas.com",
     "https://otomotif.kompas.com","https://edukasi.kompas.com",
     "https://entertainment.kompas.com","https://lifestyle.kompas.com"],
    lambda h: 'kompas.com/read' in h,
    "h1.read__title", "div.read__content p", "div.read__time"
)

semua_normal += scrape_normal(
    "CNN Indonesia", 3, 5,
    ["https://www.cnnindonesia.com/nasional","https://www.cnnindonesia.com/politik",
     "https://www.cnnindonesia.com/ekonomi","https://www.cnnindonesia.com/teknologi",
     "https://www.cnnindonesia.com/internasional","https://www.cnnindonesia.com/olahraga",
     "https://www.cnnindonesia.com/hiburan","https://www.cnnindonesia.com/gaya-hidup",
     "https://www.cnnindonesia.com/edukasi","https://www.cnnindonesia.com/indeks"],
    lambda h: 'cnnindonesia.com' in h and '/detail/' in h,
    "h1.mb-2, h1.text-cnn_black, h1", "div.detail-text p",
    "meta[property='article:published_time']"
)

semua_normal += scrape_normal(
    "Liputan6.com", 4, 5,
    ["https://www.liputan6.com/news","https://www.liputan6.com/bisnis",
     "https://www.liputan6.com/regional","https://www.liputan6.com/tekno",
     "https://www.liputan6.com/health","https://www.liputan6.com/bola",
     "https://www.liputan6.com/otomotif","https://www.liputan6.com/lifestyle",
     "https://www.liputan6.com/showbiz","https://www.liputan6.com/global"],
    lambda h: 'liputan6.com' in h and '/read/' in h,
    "h1.article-header--title, h1.read-page--header--title, h1",
    "div.article-content-body p, div[itemprop='articleBody'] p", "time"
)

semua_normal += scrape_normal(
    "Antaranews.com", 5, 5,
    ["https://www.antaranews.com/politik","https://www.antaranews.com/ekonomi",
     "https://www.antaranews.com/hukum","https://www.antaranews.com/humaniora",
     "https://www.antaranews.com/nasional","https://www.antaranews.com/olahraga",
     "https://www.antaranews.com/teknologi","https://www.antaranews.com/hiburan",
     "https://www.antaranews.com/internasional","https://www.antaranews.com/gaya-hidup"],
    lambda h: 'antaranews.com' in h and '/berita/' in h,
    "h1", "div.post-content p, div[itemprop='articleBody'] p",
    "meta[property='article:published_time']"
)

df_normal = pd.DataFrame(semua_normal)
df_normal.drop_duplicates(subset="judul", inplace=True)
df_normal = df_normal[df_normal["isi"].str.len() >= 100].reset_index(drop=True)
path_normal = os.path.join(SAVE_DIR, "dataset_berita_normal.csv")
df_normal.to_csv(path_normal, index=False, encoding="utf-8-sig")

print(f"\n{'='*55}")
print(f"✅ Scraping normal selesai!  Total: {len(df_normal)} artikel")
for s in df_normal["sumber"].unique():
    print(f"   - {s:<25}: {len(df_normal[df_normal['sumber']==s])} artikel")
print(f"💾 {path_normal}")


# ============================================================
# 3. Scraping Distorsi
# ============================================================

KEYWORDS = [
    "geger","heboh","gempar","menggemparkan","menghebohkan","viral","syok","shocking",
    "fakta tersembunyi","fakta mengejutkan","wajib tahu",
    "konspirasi","rekayasa","disembunyikan","ditutupi","dimanipulasi",
    "persekongkolan","sabotase","didalangi",
    "tanpa bukti","tanpa konfirmasi","tanpa verifikasi","disebut-sebut",
    "disinyalir","beredar luas","sumber anonim",
    "hoaks","hoax","menyesatkan","manipulasi informasi","distorsi informasi",
    "informasi menyesatkan","propaganda","pencucian otak","berita bohong",
    "berita palsu","fake news","narasi sesat","diframing",
    "pengkhianat","tukang bohong","tukang fitnah","munafik","penjilat",
    "sepanjang sejarah","sepanjang masa",
]

def parse_detik(soup):
    j = soup.select_one("h1.detail__title") or soup.select_one("h1")
    i = " ".join(p.get_text(strip=True) for p in soup.select(
        "div.detail__body-text p, div.itp_bodycontent p"))
    return (j.get_text(strip=True) if j else ""), i, get_tgl(soup)

def parse_kompas(soup):
    j = soup.select_one("h1.read__title") or soup.select_one("h1")
    i = " ".join(p.get_text(strip=True) for p in soup.select(
        "div.read__content p, div.article__content p"))
    return (j.get_text(strip=True) if j else ""), i, get_tgl(soup)

def parse_liputan6(soup):
    j = (soup.select_one("h1.article-header--title") or
         soup.select_one("h1.read-page--header--title") or soup.select_one("h1"))
    i = " ".join(p.get_text(strip=True) for p in soup.select(
        "div.article-content-body p, div[itemprop='articleBody'] p"))
    return (j.get_text(strip=True) if j else ""), i, get_tgl(soup)

def parse_antara(soup):
    j = soup.select_one("h1.post-title") or soup.select_one("h1")
    i = " ".join(p.get_text(strip=True) for p in soup.select(
        "div.post-content p, div[itemprop='articleBody'] p"))
    return (j.get_text(strip=True) if j else ""), i, get_tgl(soup)

SUMBER_DISTORSI = [
    {
        "nama" : "Detik.com",
        "home" : "https://www.detik.com/",
        "urls" : [f"https://www.detik.com/search/searchnews?query={kw.replace(' ','+')}" for kw in KEYWORDS] +
                 ["https://news.detik.com/indeks","https://news.detik.com/berita",
                  "https://hot.detik.com/celeb","https://www.detik.com/tag/viral"],
        "pola" : lambda h: bool(re.search(r'detik\.com/[^/]+/d-\d{7,}', h)),
        "parse": parse_detik,
    },
    {
        "nama" : "Kompas.com",
        "home" : "https://www.kompas.com/",
        "urls" : [f"https://search.kompas.com/search/?q={kw.replace(' ','+')}&submit=Submit" for kw in KEYWORDS] +
                 ["https://www.kompas.com/tag/viral","https://www.kompas.com/tag/hoaks",
                  "https://www.kompas.com/tren/indeks"],
        "pola" : lambda h: "kompas.com/read" in h,
        "parse": parse_kompas,
    },
    {
        "nama" : "Liputan6.com",
        "home" : "https://www.liputan6.com/",
        "urls" : [f"https://www.liputan6.com/search?q={kw.replace(' ','+')}" for kw in KEYWORDS] +
                 ["https://www.liputan6.com/tag/viral","https://www.liputan6.com/tag/hoaks",
                  "https://www.liputan6.com/news"],
        "pola" : lambda h: "liputan6.com" in h and "/read/" in h,
        "parse": parse_liputan6,
    },
    {
        "nama" : "Antaranews.com",
        "home" : "https://www.antaranews.com/",
        "urls" : [f"https://www.antaranews.com/search/?q={kw.replace(' ','+')}" for kw in KEYWORDS] +
                 ["https://www.antaranews.com/berita/terkini","https://www.antaranews.com/berita/nasional"],
        "pola" : lambda h: "antaranews.com" in h and "/berita/" in h,
        "parse": parse_antara,
    },
]

def fetch_satu(url, home, parse_fn, nama, target, hasil, seen, lck):
    with lck:
        if len(hasil) >= target or url in seen: return
    soup = get_soup(url, ref=home)
    if not soup: return
    try: judul, isi, tgl = parse_fn(soup)
    except: return
    if not judul or len(isi) < 100 or not dalam_rentang(tgl): return
    with lck:
        if len(hasil) >= target or url in seen: return
        hasil.append({"judul": judul.strip(), "isi": isi.strip(),
                      "tanggal": tgl, "sumber": nama, "url": url, "label": ""})
        seen.add(url)
        log(f"  ✓ [{len(hasil)}/{target}] [{nama}] {judul.strip()[:60]}")

def scrape_distorsi(cfg, no, total):
    nama, home, urls, pola, parse_fn = (
        cfg["nama"], cfg["home"], cfg["urls"], cfg["pola"], cfg["parse"])
    log(f"\n[{no}/{total}] Scraping distorsi {nama} (target {TARGET_DISTORSI})")
    links = kumpul_links(urls, pola, home, limit=TARGET_DISTORSI * 5)
    log(f"  📎 {len(links)} link ditemukan")
    hasil, seen, lck = [], set(), Lock()
    random.shuffle(links)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(fetch_satu, u, home, parse_fn, nama,
                          TARGET_DISTORSI, hasil, seen, lck): u for u in links}
        for f in as_completed(futs):
            with lck:
                if len(hasil) >= TARGET_DISTORSI:
                    for ff in futs: ff.cancel()
                    break
            try: f.result()
            except: pass
    log(f"  ✅ {nama}: {len(hasil)} artikel")
    return hasil

semua_distorsi = []
mulai = datetime.now()

with ThreadPoolExecutor(max_workers=4) as ex:
    futs = {ex.submit(scrape_distorsi, cfg, i+1, len(SUMBER_DISTORSI)): cfg["nama"]
            for i, cfg in enumerate(SUMBER_DISTORSI)}
    for f in as_completed(futs):
        try:
            semua_distorsi.extend(f.result())
            log(f"  ↳ total sementara: {len(semua_distorsi)} artikel")
        except Exception as e:
            log(f"  [ERROR] {futs[f]}: {e}")

df_distorsi = pd.DataFrame(semua_distorsi)
df_distorsi.drop_duplicates(subset="judul", inplace=True)
df_distorsi = df_distorsi[df_distorsi["isi"].str.len() >= 100].reset_index(drop=True)
path_distorsi = os.path.join(SAVE_DIR, "dataset_berita_distorsi.csv")
df_distorsi.to_csv(path_distorsi, index=False, encoding="utf-8-sig")

m, s = divmod(int((datetime.now()-mulai).total_seconds()), 60)
print(f"\n{'='*55}")
print(f"✅ Scraping distorsi selesai!  ⏱ {m}m {s}s  Total: {len(df_distorsi)} artikel")
for src in df_distorsi["sumber"].unique():
    print(f"   - {src:<22}: {len(df_distorsi[df_distorsi['sumber']==src])} artikel")
print(f"💾 {path_distorsi}")


# ============================================================
# 4. Gabung Dataset
# ============================================================

KOLOM = ["judul", "isi", "tanggal", "sumber", "url", "label"]

path_normal   = os.path.join(SAVE_DIR, "dataset_berita_normal.csv")
path_distorsi = os.path.join(SAVE_DIR, "dataset_berita_distorsi.csv")
path_gabung   = os.path.join(SAVE_DIR, "dataset_gabungan.csv")

df_n = pd.read_csv(path_normal,   encoding="utf-8-sig")
df_d = pd.read_csv(path_distorsi, encoding="utf-8-sig")

for kolom in KOLOM:
    if kolom not in df_n.columns: df_n[kolom] = ""
    if kolom not in df_d.columns: df_d[kolom] = ""

df_n["label"] = ""
df_d["label"] = ""

df_gabung = pd.concat([df_n[KOLOM], df_d[KOLOM]], ignore_index=True)
sebelum = len(df_gabung)
df_gabung.drop_duplicates(subset="judul", inplace=True)
df_gabung = df_gabung[df_gabung["isi"].astype(str).str.len() >= 100].reset_index(drop=True)
df_gabung.to_csv(path_gabung, index=False, encoding="utf-8-sig")

print(f"\n{'='*55}")
print(f"✅ Penggabungan selesai!")
print(f"   Duplikat dihapus : {sebelum - len(df_gabung)}")
print(f"   Total akhir      : {len(df_gabung)} artikel")
print(f"💾 {path_gabung}")


# ============================================================
# 5. Visualisasi Dataset
# ============================================================

df_gabung = pd.read_csv(path_gabung, encoding="utf-8-sig")
per_sumber = df_gabung["sumber"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Kiri: per sumber ─────────────────────────────────────────
axes[0].bar(per_sumber.index, per_sumber.values, color="steelblue")
axes[0].set_title("Jumlah Artikel per Sumber")
axes[0].set_xlabel("Sumber")
axes[0].set_ylabel("Jumlah Artikel")
axes[0].tick_params(axis="x", rotation=20)
for i, v in enumerate(per_sumber.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# ── Kanan: normal vs distorsi vs total ───────────────────────
n_normal   = len(df_n)
n_distorsi = len(df_d)
n_gabung   = len(df_gabung)

axes[1].bar(["Normal", "Distorsi", "Total Gabungan"],
            [n_normal, n_distorsi, n_gabung],
            color=["steelblue", "tomato", "seagreen"])
axes[1].set_title("Jumlah Dataset Gabungan")
axes[1].set_ylabel("Jumlah Artikel")
for i, v in enumerate([n_normal, n_distorsi, n_gabung]):
    axes[1].text(i, v + 5, str(v), ha="center", fontweight="bold")

plt.suptitle("Ringkasan Dataset Scraping", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "visualisasi_dataset.png"), dpi=150)
plt.show()

print(f"\n📊 Visualisasi disimpan → {os.path.join(SAVE_DIR, 'visualisasi_dataset.png')}")
print(f"   Normal   : {n_normal} artikel")
print(f"   Distorsi : {n_distorsi} artikel")
print(f"   Gabungan : {n_gabung} artikel")